# Combining MC + data

Continues from `step1_new_analysis_mc_only.ipynb` -- run that first.
Covers adding on-beam & off-beam data, and POT/gate accounting.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

import pandas as pd
from cafpybara import core

# Continues step1 -- import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

In [ ]:
# Define once, used throughout this notebook (continues step1's
# MC_FILE/VAR/BINS convention). MC_FILE/DATA_FILE/OFFBEAM_FILE must
# also be produced by cafpyana first, e.g. via
# `run_df_maker.py -c configs/hnl_data_nopreselect_savepfp.py -l <input.list> -o <output_name>`
# (no preselection, all PFPs saved -- see cafpyana's hnl_nuee_nupi0 README.
#
# NOTE: section 6's `mcstat` call uses a `file_idx` column -- that
# assumes each of these is really a *list* of files concatenated
# together (see `concat_hdf.py`).
MC_FILE = "your_mc_file.h5"
DATA_FILE = "your_data_file.h5"
OFFBEAM_FILE = "your_offbeam_file.h5"
VAR = "your_variable"
BINS = [0, 1, 2, 3]

## 1. `io.py` -- `load_data` (on-beam)

* TODO: none -- `load_data` reuses the `REC_KEY` set on `io.py`.
* Arguments: `onbeam=True` by default, to load on-beam data.
* Returns:
  * `df`: dataframe
  * `pot`: on-beam POT
  * `ngates`: on-beam gate count

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    data_df, data_pot, data_ngates = ana.load_data(DATA_FILE)
    print("Data POT:   ", data_pot)
    print("Data gates: ", data_ngates)
except Exception as e:
    print("No real data file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 2. `preprocess.py` -- `preprocess_data` (on-beam) (Optional)

* TODO: add your data-only fixes to `preprocess_data` (e.g. a
  run/subrun-quality cut, data-specific timing calibration).

Example -- HNL/pi0's Data BNB timing calibration
(`analyses/hnlpi0/preprocess.py`):

```python
def preprocess_databnb(df):
    df = preprocess_data(df)
    df = fix_databnb_timing_calibration(df)  # drops bad-period rows,
                                              # corrects each good period
    df = add_variables(df)
    return df
```

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    data_df, data_pot, data_ngates = ana.load_data(DATA_FILE, preprocess_fn=ana.preprocess.preprocess_data)
    print("Data POT:   ", data_pot)
    print("Data gates: ", data_ngates)
except Exception as e:
    print("No real data file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 3. `io.py` -- `load_data` (off-beam)

* TODO:
  * pass `onbeam=False` to load an off-beam file instead.
  * add a dedicated `"offbeam"` entry to `signal_categories` in
    `analysis.py`, and pass it as `offbeam_signal_value`.
* Arguments: `onbeam=False`; `offbeam_signal_value` -- signal
  category to stamp on every off-beam row.
* Returns:
  * `df`: dataframe
  * `pot`: off-beam POT (always `0.0`)
  * `ngates`: off-beam gate count

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False,
        offbeam_signal_value=ana.signal_dict["offbeam"],  # TODO above -- add this category first
    )
    print("Offbeam POT:   ", offbeam_pot)
    print("Offbeam gates: ", offbeam_ngates)
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 4. `preprocess.py` -- `preprocess_data` (off-beam) (Optional)

* TODO: add your off-beam-only fixes (e.g. off-beam-specific timing
  calibration), similar to section 2's on-beam `preprocess_data`.

Example -- HNL/pi0's Data Offbeam+Light timing calibration
(`analyses/hnlpi0/preprocess.py`):

```python
def preprocess_dataoff(df):
    df = preprocess_data(df)
    df = fix_timing_calibration(df, period=tc.offbeam_period_calib,
                                 t0_offset=tc.offbeam_offset_calib,
                                 ifData=True)
    df = add_variables(df)
    return df
```

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False, offbeam_signal_value=ana.signal_dict["offbeam"],
        preprocess_fn=ana.preprocess.preprocess_data,
    )
    print("Offbeam POT:   ", offbeam_pot)
    print("Offbeam gates: ", offbeam_ngates)
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 5. Off-beam scaling

* TODO: double-check `f` is the most up-to-date fudge factor.

Scale off-beam data to MC BNB POT. See SBN-docdb-41013 &
SBN-docdb-43255.

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False, offbeam_signal_value=ana.signal_dict["offbeam"],
    )
    data_df, data_pot, data_ngates = ana.load_data(DATA_FILE)
    mc_df, mc_pot, mc_ngen = ana.load_mc(MC_FILE)

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates
    print("Offbeam scale: ", scale_offbeam)
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 6. Merge off-beam into the MC

* TODO:
  * weight: `1.0` on `mc_df`, `scale_offbeam` (section 5) on
    `offbeam_df`.
  * concat: `pd.concat` them so `plot_mc_data` stacks off-beam
    alongside the other MC backgrounds -- on-beam `data_df` stays
    the separate overlay.
* `ana.mcstat` adds bootstrapped MC statistical-uncertainty universes
  to `mc_df` before the merge.

Example -- HNL/pi0's merge
(`analyses/hnlpi0/examples/hnl_plotting.ipynb`).

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False, offbeam_signal_value=ana.signal_dict["offbeam"],
        cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    mc_df, mc_pot, mc_ngen = ana.load_mc(
        MC_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_mc,
    )

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample', 'file_idx']
    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 7. `plotting.py` -- `plot_mc_data`

* TODO:
  * none new -- reuses `signal_categories` in `analysis.py`.
  * inspect `plot_mc_data`/`plot_var`'s docstring for plotting
    options (labels, systematics, legend, etc.).
* Composes `plot_var` (MC stack) with a data overlay and a ratio panel.
* Reuses `mc_df`/`data_df`/`mc_pot`/`data_pot` from section 6 --
  `mc_df` is already the merged MC+off-beam stack, and `plot_var`
  picks up its `weights_mc` column automatically.

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    # Reuses mc_df/data_df/mc_pot/data_pot from section 6 -- mc_df is
    # the merged MC+off-beam stack, with weights_mc set on both.
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)
except Exception as e:
    print("No real MC/data file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")